# 04 · Supervised fine-tuning (SFT) with TRL + LoRA

**Track:** Training LLMs & Frontier Research · **Runtime (CPU default):** ~3 minutes · **GPU:** optional (full-size cell)

A pre-trained base model is a brilliant autocomplete engine and a terrible assistant: ask it a question and it may
continue with three more questions, a forum signature, or a page of unrelated text. **Supervised fine-tuning** is
the first post-training step (step 1 of InstructGPT's three-step recipe): show the model a few thousand (prompt,
ideal response) pairs and train it, with ordinary next-token cross-entropy, *only on the response tokens*. In this
notebook we SFT a 135M-parameter base model (SmolLM2-135M) on 40 hand-written examples with Hugging Face **TRL**, using
**LoRA** so that only ~3% of the weights train. It takes a couple of minutes on a laptop CPU, and the before/after
difference in *format* is striking, even though the model learns little new knowledge.

> **Source & license credit.** Training uses Hugging Face **[TRL](https://github.com/huggingface/trl)**
> (`SFTTrainer`, Apache-2.0), **[PEFT](https://github.com/huggingface/peft)** (LoRA, Apache-2.0),
> **[transformers](https://github.com/huggingface/transformers)** and **[datasets](https://github.com/huggingface/datasets)**
> (Apache-2.0). Base model: **[HuggingFaceTB/SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M)**
> (Apache-2.0), downloaded at runtime (~270 MB) and not redistributed. The 40-example instruction dataset
> [`research/data/sft_tiny.jsonl`](../data/sft_tiny.jsonl) was written for this repo (original; same license as the
> repo). The overall flow mirrors the TRL SFT documentation examples. See
> [`research/THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

## Learning goals

1. Explain what SFT changes (format, style, instruction following) and what it mostly does *not* change (knowledge).
2. Build a **prompt–completion** dataset and verify that the loss is **masked to completion tokens only**.
3. Explain **LoRA**: $W' = W + \frac{\alpha}{r} BA$, with $B \in \mathbb{R}^{d\times r}$ and $A \in \mathbb{R}^{r\times k}$, and why it cuts optimiser memory.
4. Run `trl.SFTTrainer` end to end on CPU and measure before/after behaviour with a simple, honest metric.

## Background

* Ouyang et al., *Training language models to follow instructions with human feedback* (InstructGPT), OpenAI 2022.
  [arXiv:2203.02155](https://arxiv.org/abs/2203.02155): SFT → reward model → PPO.
* Wei et al., *Finetuned Language Models Are Zero-Shot Learners* (FLAN), 2021. [arXiv:2109.01652](https://arxiv.org/abs/2109.01652)
* Zhou et al., *LIMA: Less Is More for Alignment*, 2023. [arXiv:2305.11206](https://arxiv.org/abs/2305.11206):
  the "superficial alignment hypothesis", which says SFT mostly teaches *format*, and 1,000 curated examples go a long way.
* Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, 2021. [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)
* Dettmers et al., *QLoRA*, 2023. [arXiv:2305.14314](https://arxiv.org/abs/2305.14314)
* Allal et al., *SmolLM2: When Smol Goes Big*, 2025. [arXiv:2502.02737](https://arxiv.org/abs/2502.02737)

**The SFT objective.** For a prompt $x$ and target response $y = (y_1,\dots,y_m)$:
$$\mathcal{L}_{SFT}(\theta) = -\sum_{t=1}^{m} \log \pi_\theta(y_t \mid x, y_{<t}).$$
Same cross-entropy as pre-training; the only differences are the data and the mask over prompt tokens.

In [ ]:
# --- Setup: make research/src importable (from research/, research/notebooks/, or a Colab clone) ---
import sys, pathlib
for _p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]:
    _src = next((c for c in (_p / "src", _p / "research" / "src") if (c / "lab_config.py").exists()), None)
    if _src:
        sys.path.insert(0, str(_src))
        break
else:
    raise RuntimeError("Could not find research/src -- run this notebook from inside the repo.")

from lab_config import CFG, pick, set_seed, heavy_cell, read_text, quiet_hf_logging

FULL_SIZE = CFG.full_size   # <- set True here (or export LAB_FULL_SIZE=1) for the full-size run
CFG.full_size = FULL_SIZE
set_seed()
print(CFG.summary())
quiet_hf_logging()

In [ ]:
import json, time
import torch
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

BASE_MODEL = pick("HuggingFaceTB/SmolLM2-135M", "HuggingFaceTB/SmolLM2-360M")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(CFG.device)
print(f"{BASE_MODEL}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

rows = [json.loads(l) for l in (CFG.data_dir / "sft_tiny.jsonl").read_text().splitlines()]
eval_prompts = json.loads((CFG.data_dir / "sft_eval_prompts.json").read_text())
print(len(rows), "training examples;", len(eval_prompts), "held-out prompts")
print(json.dumps(rows[2], indent=1, ensure_ascii=False))

## 1 · The prompt template

The base model has never seen a chat format, so we invent a simple one (Alpaca-style). Every training response also
ends with a `TL;DR:` line. That is a deliberately **visible stylistic signature**: if SFT works, the model will start
producing it on prompts it has never seen.

In [ ]:
def format_prompt(instruction: str) -> str:
    return f"### Instruction:\n{instruction}\n\n### Response:\n"

@torch.no_grad()
def generate(m, instruction, max_new_tokens=80):
    enc = tokenizer(format_prompt(instruction), return_tensors="pt").to(CFG.device)
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
                     pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
    new = out[0, enc["input_ids"].shape[1]:]
    stopped = bool((new == tokenizer.eos_token_id).any())
    return tokenizer.decode(new, skip_special_tokens=True).strip(), stopped

model.eval()
t0 = time.time()
before = {p: generate(model, p) for p in eval_prompts}
print(f"generated {len(before)} baseline answers in {time.time() - t0:.0f}s\n")
for p in eval_prompts[:3]:
    print(f"### {p}\n{before[p][0][:300]}\n[stopped by itself: {before[p][1]}]\n")

## 2 · Dataset → tokens, with the loss mask

TRL's **prompt–completion** format (`{"prompt": ..., "completion": ...}`) tells `SFTTrainer` which part is the prompt.
By default (`completion_only_loss=True`) prompt tokens get label `-100`, which PyTorch's cross-entropy ignores, and an
EOS token is appended to each completion so the model learns **when to stop**.

In [ ]:
train_ds = Dataset.from_list([{"prompt": format_prompt(r["instruction"]), "completion": r["response"]} for r in rows])

lora_cfg = LoraConfig(
    r=pick(16, 32), lora_alpha=pick(32, 64), lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
args = SFTConfig(
    output_dir=str(CFG.output_dir / "sft_run"),
    max_steps=pick(40, 300),                 # 40 steps x 4 = 160 examples = 4 epochs over our 40
    per_device_train_batch_size=4,
    learning_rate=pick(1e-3, 2e-4),          # LoRA tolerates (and on tiny budgets needs) a high LR
    lr_scheduler_type="cosine", warmup_steps=pick(4, 20),
    max_length=256,
    logging_steps=5,
    save_strategy="no", report_to=[], disable_tqdm=True,
    seed=CFG.seed, use_cpu=not CFG.has_gpu,
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, processing_class=tokenizer, peft_config=lora_cfg)

ex = trainer.train_dataset[0]
labels = ex.get("labels")
if labels is None:   # some TRL versions keep a completion_mask instead of pre-built labels
    labels = [t if m else -100 for t, m in zip(ex["input_ids"], ex["completion_mask"])]
n_masked = sum(1 for l in labels if l == -100)
print(f"example 0: {len(ex['input_ids'])} tokens, {n_masked} masked (prompt), {len(labels) - n_masked} trained (completion)")
print("trained-on text:", repr(tokenizer.decode([t for t, l in zip(ex["input_ids"], labels) if l != -100])))

## 3 · LoRA: train a low-rank update instead of the full matrix

For a frozen weight $W\in\mathbb{R}^{d\times k}$, LoRA learns $\Delta W = \frac{\alpha}{r}BA$ with rank $r \ll \min(d,k)$.
$B$ starts at zero, so training begins exactly at the base model. Adam keeps two extra moments per *trainable*
parameter, so training 3% of the weights cuts optimiser memory by roughly 30×. That is the main reason LoRA/QLoRA made
fine-tuning possible on consumer GPUs.

In [ ]:
trainer.model.print_trainable_parameters()

In [ ]:
t0 = time.time()
trainer.train()
print(f"SFT finished in {time.time() - t0:.0f}s")

logs = [h for h in trainer.state.log_history if "loss" in h]
plt.figure(figsize=(5, 3))
plt.plot([h["step"] for h in logs], [h["loss"] for h in logs], "o-", label="train loss (completion tokens)")
if "mean_token_accuracy" in logs[0]:
    plt.twinx().plot([h["step"] for h in logs], [h["mean_token_accuracy"] for h in logs], "s--", c="C1", alpha=0.6)
plt.xlabel("step"); plt.title("SFT loss"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4 · Before vs. after on held-out prompts

We use two crude but honest metrics on the 8 held-out prompts:
* **stops by itself**: the model emits EOS within 80 tokens instead of rambling on;
* **uses the TL;DR format**: the learned stylistic signature appears.

Neither says the answers are *correct*. Read them yourself.

In [ ]:
sft_model = trainer.model
sft_model.eval()
after = {p: generate(sft_model, p) for p in eval_prompts}

def summary(res):
    return (sum(s for _, s in res.values()) / len(res), sum("TL;DR:" in t for t, _ in res.values()) / len(res))

for name, res in [("base", before), ("SFT ", after)]:
    stop_rate, tldr_rate = summary(res)
    print(f"{name}: stops by itself {stop_rate:.0%} | uses TL;DR format {tldr_rate:.0%}")

for p in eval_prompts:
    print("=" * 100)
    print("PROMPT:", p)
    print("--- base ---\n" + before[p][0][:280].replace("\n", " ⏎ "))
    print("--- SFT  ---\n" + after[p][0][:280].replace("\n", " ⏎ "))

**What to look for.** After 40 steps on 40 examples, the model typically adopts the format (answering directly,
ending with a `TL;DR:` line and stopping), which is consistent with LIMA's superficial alignment hypothesis. Factual
accuracy is whatever the 135M base model already knew, and on the refusal-style held-out prompt (reading a
coworker's messages) a single refusal example in training is rarely enough. Those are the jobs of more data, more
careful data, and preference tuning (notebook 05).

## 5 · Save the adapter

LoRA adapters are tiny compared with the model, which is why "one base model + many adapters" serving is popular.

In [ ]:
adapter_dir = CFG.output_dir / "sft_lora_adapter"
sft_model.save_pretrained(adapter_dir)
size_mb = sum(f.stat().st_size for f in adapter_dir.rglob("*") if f.is_file()) / 1e6
print(f"adapter saved to research/outputs/{adapter_dir.name} ({size_mb:.1f} MB vs ~{sum(p.numel() for p in model.parameters()) * 4 / 1e6:.0f} MB for full fp32 weights)")

# For deployment you can fold the update into the base weights: W <- W + (alpha/r) * B @ A
merged = sft_model.merge_and_unload()
print("merged model type:", type(merged).__name__)

## 6 · (Optional, heavy) a real SFT run

A realistic small-scale SFT uses tens of thousands of chat examples with a chat template, for example the open
[`HuggingFaceTB/smoltalk`](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) dataset (check its card for license
terms), on a GPU. This cell downloads a 2k-example slice and fine-tunes `SmolLM2-360M` with the model's own chat
template. **GPU recommended; skipped by default.**

In [ ]:
if heavy_cell("SFT of SmolLM2-360M on a 2k-example smoltalk slice", needs_gpu=True):
    from datasets import load_dataset
    chat_ds = load_dataset("HuggingFaceTB/smoltalk", "everyday-conversations", split="train[:2000]")
    big_tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")  # borrow the chat template
    big = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-360M")
    big_args = SFTConfig(output_dir=str(CFG.output_dir / "sft_big"), num_train_epochs=1, per_device_train_batch_size=8,
                         learning_rate=1e-4, max_length=1024, logging_steps=20, report_to=[], save_strategy="no",
                         bf16=torch.cuda.is_bf16_supported())
    SFTTrainer(model=big, args=big_args, train_dataset=chat_ds, processing_class=big_tok).train()

## Your turn

1. **Ablate the loss mask.** Set `completion_only_loss=False` in `SFTConfig` (train on prompt tokens too). Does the
   format still transfer? Does the model start generating `### Instruction:` blocks of its own?
2. **Rank.** Try LoRA `r ∈ {1, 4, 16, 64}`. At what rank does the TL;DR adoption rate saturate?
3. **Teach a refusal.** Add 5 more refusal examples (polite, offers a safe alternative) and re-check the held-out
   privacy prompt. Then check whether the model starts refusing *benign* prompts too (over-refusal).
4. **Full fine-tune.** Drop `peft_config` and lower the LR to 5e-5. Compare speed, memory and results.

## Beyond the paper — a mini research question

*How many examples does it take to learn a format?* LIMA argues ~1,000 curated examples suffice for a strong assistant.
At toy scale, measure TL;DR adoption on the held-out prompts as a function of training-set size (2, 5, 10, 20, 40
examples, with a fixed number of *steps*) and of model size (135M vs. 360M). Is there a sharp threshold? Does a larger
base model need fewer examples, which is what the superficial alignment hypothesis would predict?

## References

* Ouyang, L. et al. (2022). *Training language models to follow instructions with human feedback.* [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)
* Wei, J. et al. (2021). *Finetuned Language Models Are Zero-Shot Learners.* [arXiv:2109.01652](https://arxiv.org/abs/2109.01652)
* Zhou, C. et al. (2023). *LIMA: Less Is More for Alignment.* [arXiv:2305.11206](https://arxiv.org/abs/2305.11206)
* Hu, E. et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models.* [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)
* Dettmers, T. et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs.* [arXiv:2305.14314](https://arxiv.org/abs/2305.14314)
* Allal, L. B. et al. (2025). *SmolLM2: When Smol Goes Big.* [arXiv:2502.02737](https://arxiv.org/abs/2502.02737)
* von Werra, L. et al. *TRL: Transformer Reinforcement Learning* (Apache-2.0). https://github.com/huggingface/trl